# the organization Phase 1 KPI — Data Cleaning & Correlation Prep
**Purpose:** clean the "Phase 1 - KPI Summary" tab and prepare it for correlation
analysis between the *input-side* KPI ratios (ETA Achievement, Rework Rate, Idle
Rate, Effective Utilisation, Meeting Overhead Ratio, Req. Understanding Ratio,
Avg Time Per Task).

**Not covered here:** CSAT, # of Defects, # of Escalations, and KPI Achievement
are still blank in the source file (pending Capability Manager entry), so this
notebook cannot correlate inputs against those outputs yet. Re-run once they're
filled in.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


## 2. Upload the file
Run this cell in Colab and select
`the organization_-_Phase_1_KPI_Table_-_May_2026.xlsx` when prompted.
If you're running locally instead of Colab, skip this cell and just set
`FILENAME` to the path of the file on your machine.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    FILENAME = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab — set this manually
    FILENAME = "the organization_-_Phase_1_KPI_Table_-_May_2026.xlsx"

print("Using file:", FILENAME)


## 3. Load the KPI Summary sheet

In [ ]:
df_raw = pd.read_excel(FILENAME, sheet_name="Phase 1 - KPI Summary")
print("Shape:", df_raw.shape)
df_raw.head(10)


## 4. Initial inspection
Check dtypes and get a first look at where "-" strings, blanks, or unexpected
types show up (Excel sheets often mix numbers with placeholder text like "-").

In [ ]:
df_raw.info()


In [ ]:
# Columns that contain non-numeric placeholders (e.g. "-") mixed with numbers
for col in df_raw.columns:
    if df_raw[col].dtype == object:
        uniques = df_raw[col].dropna().unique()
        print(f"{col}: {uniques[:8]}")


## 5. Missing value check (on the full sheet, before any cleaning)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(1)
}).sort_values("missing_pct", ascending=False)

missing_summary


In [ ]:
# Columns that are 100% empty — these are the CM-entered output KPIs,
# not yet available for analysis.
fully_empty_cols = missing_summary[missing_summary["missing_pct"] == 100.0].index.tolist()
print("Fully empty columns (not usable yet):")
for c in fully_empty_cols:
    print(" -", c)


## 6. Clean the data

Steps:
1. Drop the fully-empty output KPI columns for now (kept in a list above so you
   can re-add them once populated).
2. Replace `"-"` placeholder strings with `NaN` so numeric columns are actually
   numeric.
3. Drop rows with zero `Total Billable Hrs` — these are teammates with no
   logged activity that month, so their ratios (0/0-type values) aren't
   meaningful for correlation.
4. Coerce the relevant columns to numeric dtype.


In [ ]:
df = df_raw.drop(columns=fully_empty_cols)

# Replace "-" placeholders with NaN across the whole frame
df = df.replace("-", np.nan)

# Drop zero-activity rows
before = len(df)
df = df[df["Total Billable Hrs"] > 0].reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} zero-activity row(s); {after} rows remain.")


In [ ]:
input_cols = [
    "ETA Achievement",
    "Rework Rate",
    "Req. Understanding Ratio",
    "Meeting Overhead Ratio",
    "Idle Rate",
    "Effective Utilisation",
    "Avg Time Per Task (hrs)",
]

for c in input_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[input_cols].describe()


## 7. Missing value check — input columns only
Confirm there are no gaps left in the columns we're about to correlate.

In [ ]:
df[input_cols].isna().sum().to_frame("missing_count")


## 8. ⚠️ Formula-overlap caveat
`ETA Achievement` and `Effective Utilisation` share components by construction:

- `ETA Achievement = (Production + QC) / Available Hrs`
- `Effective Utilisation = (Production + QC + Rework + Req. Understanding) / Available Hrs`

Because Effective Utilisation's numerator *contains* ETA Achievement's numerator,
any correlation between the two is a mechanical artifact of the formulas, not an
independent relationship. The cell below flags this pair so it isn't
mis-reported as a finding — it doesn't remove the columns, since both are still
useful individually.

In [ ]:
overlapping_pairs = [("ETA Achievement", "Effective Utilisation")]
print("Flagged as formula-overlap, exclude from findings:")
for a, b in overlapping_pairs:
    print(f" - {a}  <->  {b}")


## 9. Correlation-ready dataset
This is the cleaned table to use for analysis: one row per teammate/month,
zero-activity rows removed, only numeric input-side ratios retained.

In [ ]:
corr_ready = df[["Teammate ID", "Capability Practice", "Client Code", "Period"] + input_cols].copy()
corr_ready.head(10)


## 10. Correlation analysis (Spearman)

In [ ]:
corr_matrix = corr_ready[input_cols].corr(method="spearman")

def spearman_pvalues(data, cols):
    pvals = pd.DataFrame(np.ones((len(cols), len(cols))), columns=cols, index=cols)
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if i < j:
                sub = data[[a, b]].dropna()
                if len(sub) >= 4:
                    _, p = spearmanr(sub[a], sub[b])
                else:
                    p = np.nan
                pvals.loc[a, b] = p
                pvals.loc[b, a] = p
    return pvals

pval_matrix = spearman_pvalues(corr_ready, input_cols)

print("Spearman correlation matrix:")
display(corr_matrix.round(2))
print("\np-value matrix:")
display(pval_matrix.round(3))


In [ ]:
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, square=True, cbar_kws={"label": "Spearman rho"})
plt.title("Spearman correlations — input-side KPI ratios (May 2026)")
plt.tight_layout()
plt.show()


## 11. Significant pairs (p < 0.05), overlap pair excluded
Quick shortlist of relationships worth writing up, with the mechanically
overlapping pair filtered out.

In [ ]:
results = []
cols = input_cols
for i, a in enumerate(cols):
    for j, b in enumerate(cols):
        if i < j and (a, b) not in overlapping_pairs and (b, a) not in overlapping_pairs:
            rho = corr_matrix.loc[a, b]
            p = pval_matrix.loc[a, b]
            n = corr_ready[[a, b]].dropna().shape[0]
            results.append({"var_1": a, "var_2": b, "rho": rho, "p_value": p, "n": n})

sig = pd.DataFrame(results).sort_values("p_value")
sig[sig["p_value"] < 0.05]


## 12. Nonzero-observation check for zero-inflated columns
Some input columns (Idle Rate, Rework Rate, Req. Understanding Ratio) are zero
for most rows. A correlation involving one of these can look statistically
significant while actually being driven by only a handful of nonzero rows.
This cell counts, for every significant pair, how many rows are nonzero in
each variable, and flags any pair where that count falls below a threshold.

In [ ]:
# How many nonzero observations does each input column actually have?
nonzero_counts = (corr_ready[input_cols] != 0).sum().rename("nonzero_count")
nonzero_counts_pct = ((corr_ready[input_cols] != 0).mean() * 100).round(1).rename("nonzero_pct")

nonzero_summary = pd.concat([nonzero_counts, nonzero_counts_pct], axis=1)
nonzero_summary


In [ ]:
# Attach nonzero counts to the significant-pairs table and flag thin ones
THIN_THRESHOLD = 5  # fewer than this many nonzero rows = treat with caution

sig = sig.copy()
sig["nonzero_n_var_1"] = sig["var_1"].map(nonzero_counts)
sig["nonzero_n_var_2"] = sig["var_2"].map(nonzero_counts)
sig["min_nonzero_n"] = sig[["nonzero_n_var_1", "nonzero_n_var_2"]].min(axis=1)
sig["thin_result"] = sig["min_nonzero_n"] < THIN_THRESHOLD

sig_flagged = sig[sig["p_value"] < 0.05].sort_values("p_value")
sig_flagged


In [ ]:
# Split into results you can report with confidence vs. ones needing a caveat
robust_results = sig_flagged[~sig_flagged["thin_result"]]
thin_results = sig_flagged[sig_flagged["thin_result"]]

print("Robust significant pairs (safe to report as-is):")
display(robust_results)

print(f"\nThin significant pairs (driven by fewer than {THIN_THRESHOLD} nonzero rows — report with caveat, if at all):")
display(thin_results)


## 13. Export the cleaned dataset
Saves a CSV you can reuse later (e.g. once CSAT / KPI Achievement are filled in,
you can merge on `Teammate ID` + `Period`).

In [ ]:
out_path = "cleaned_kpi_inputs_may2026.csv"
corr_ready.to_csv(out_path, index=False)
print("Saved:", out_path)

try:
    from google.colab import files as colab_files
    colab_files.download(out_path)
except ImportError:
    pass
